# Week 8 — Cloud Computing
### From "it runs on my laptop" to "it runs for everyone"

For seven weeks we built models, pipelines and AI tools that run on one machine.
Real data work runs in the **cloud**. This week is about *ideas*, not accounts or bills.
We use small Python simulations to see **why** the cloud works the way it does.

| Time | Chapter | The question it answers |
|---|---|---|
| 0–5 | Opening | Where has our code been running all along? |
| 5–13 | 1. Own vs rent | Why do companies rent computers instead of buying them? |
| 13–23 | 2. IaaS, PaaS, SaaS | How much does the provider manage for you? |
| 23–33 | 3. Scaling | What happens when 50× more users show up? |
| 33–38 | 4. Cloud limitations | When is the cloud the wrong choice? |
| 38–46 | 5. The provider landscape | AWS, GCP, Azure… how do I read a job description? |
| 46–56 | 6. Why Supabase | What can a table do that a file can't? |
| 56–60 | Wrap-up | What we build in Weeks 9–11 |

**No cloud account needed.** Everything here runs in this notebook.

## Setup — run this first

In [ ]:
import math, sqlite3, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
print("Ready. No cloud account needed.")

---
## Opening — where has our code been running?  *(0–5 min)*

In [ ]:
import platform, os
print("Machine:   ", platform.node())
print("System:    ", platform.system(), platform.release())
print("CPU cores: ", os.cpu_count())

---
## Chapter 1 — Own vs rent  *(5–13 min)*

**The core idea:** instead of buying and running your own servers, you **rent** computing from a provider and **pay for what you use**.

Same logic as a car: if you drive every day, buying might make sense. If you need a van twice a year, you rent one.

Let's compare buying a GPU server with renting one, over 3 years. *(Made-up round prices, to show the shape.)*

In [ ]:
BUY_PRICE = 30_000          # buy a GPU server up front ($)
BUY_MONTHLY = 200           # power, cooling, repairs, per month ($)
RENT_PER_HOUR = 3.00        # rent the same GPU by the hour ($)
MONTHS = 36

usage = {"Train a model a few times a month (10 h/month)": 10,
         "Busy team (200 h/month)": 200,
         "Running 24/7 (730 h/month)": 730}

rows = []
for label, hours in usage.items():
    buy = BUY_PRICE + BUY_MONTHLY * MONTHS
    rent = RENT_PER_HOUR * hours * MONTHS
    rows.append({"How much you use it": label, "Buy ($, 3 yrs)": f"{buy:,.0f}",
                 "Rent ($, 3 yrs)": f"{rent:,.0f}", "Cheaper": "rent" if rent < buy else "buy"})
pd.DataFrame(rows)

In [ ]:
months = np.arange(0, MONTHS + 1)
plt.figure(figsize=(7, 3.5))
plt.plot(months, BUY_PRICE + BUY_MONTHLY * months, lw=3, label="Buy (pay up front)")
for label, hours in usage.items():
    plt.plot(months, RENT_PER_HOUR * hours * months, label="Rent: " + label.split("(")[1].rstrip(")"))
plt.xlabel("Months"); plt.ylabel("Total cost ($)"); plt.title("Own vs rent over 3 years")
plt.legend(fontsize=8); plt.grid(alpha=.3); plt.show()

---
## Chapter 2 — IaaS, PaaS, SaaS: how much do *you* manage?  *(13–23 min)*

One question defines the whole spectrum: **how much does the provider manage for you?**

Think pizza 🍕: build your own kitchen → rent a kitchen → bring just your recipe → order delivery.

In [ ]:
layers = ["Your app's data", "Your code", "Runtime (Python, libraries)", "Operating system",
          "Servers & storage", "Network", "Building, power, cooling"]
who = {
    "On-premises\n(own kitchen)":  ["You"] * 7,
    "IaaS\n(rent a kitchen)":       ["You", "You", "You", "You", "Provider", "Provider", "Provider"],
    "PaaS\n(bring your recipe)":    ["You", "You", "Provider", "Provider", "Provider", "Provider", "Provider"],
    "SaaS\n(order delivery)":       ["You*", "Provider", "Provider", "Provider", "Provider", "Provider", "Provider"],
}
table = pd.DataFrame(who, index=layers)

def colour(v):
    return "background-color:#f6c9a8" if v.startswith("You") else "background-color:#bfe3d6"
table.style.map(colour)

*\* With SaaS you still own your data and your account, just not any of the machinery.*

| Model | You manage | Example |
|---|---|---|
| **IaaS** — Infrastructure | OS, software, security, your code | AWS EC2, Azure VMs |
| **PaaS** — Platform | just your code (and data) | AWS Elastic Beanstalk, Azure App Service |
| **SaaS** — Software | nothing, you just use it | Gmail, Dropbox, Google Docs |

### Your turn: where do tools you've already used sit?
Change the guesses, then run the cell.

In [ ]:
my_guesses = {
    "Gmail": "SaaS",
    "Google Colab": "SaaS",
    "GitHub": "SaaS",
    "OpenAI API": "PaaS",
    "AWS EC2": "IaaS",
    "Render (deploy from a Git repo)": "PaaS",
}

answers = {
    "Gmail": ("SaaS", "You just use the app."),
    "Google Colab": ("SaaS or PaaS", "A ready-made notebook app, but you bring your own code to run."),
    "GitHub": ("SaaS", "You use the app to store code. (GitHub Actions, which runs your code, is more like PaaS.)"),
    "OpenAI API": ("SaaS or PaaS", "You don't manage any servers or models. You call it from your code."),
    "AWS EC2": ("IaaS", "A raw virtual machine. You install and secure everything."),
    "Render (deploy from a Git repo)": ("PaaS", "You bring code, they run and scale it."),
}
for tool, guess in my_guesses.items():
    right, why = answers[tool]
    mark = "✅" if guess in right else "🤔"
    print(f"{mark} {tool:<34} you: {guess:<5} answer: {right:<12} {why}")

---
## Chapter 3 — Scaling: launch day  *(23–33 min)*

Our weather app normally has about **1,000 users**. Today we launch, and for a few hours **50,000** show up.

Two ways to cope:
- **Vertical scaling** — make one machine bigger. Simple, but every machine has a maximum size.
- **Horizontal scaling** — add more machines and split the work. Add them when traffic rises, remove them when it drops.

Let's simulate one day. *(Made-up capacities: one normal server handles 5,000 users; the biggest single machine handles 20,000.)*

In [ ]:
hours = np.arange(24)
users = 1_000 + 600 * np.sin((hours - 6) / 24 * 2 * np.pi)          # a normal day
users[10:15] = [20_000, 50_000, 45_000, 30_000, 12_000]               # launch spike at 10am
users = users.round().astype(int)

SERVER_CAPACITY = 5_000     # users one normal server can handle
BIGGEST_MACHINE = 20_000    # the ceiling for vertical scaling

vertical_served = np.minimum(users, BIGGEST_MACHINE)
servers_needed = np.maximum(1, np.ceil(users / SERVER_CAPACITY)).astype(int)
horizontal_capacity = servers_needed * SERVER_CAPACITY

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(hours, users, color="#c9d6e3", label="Users wanting in")
ax.step(hours, horizontal_capacity, where="mid", lw=2, label="Horizontal: capacity (adds servers)")
ax.axhline(BIGGEST_MACHINE, color="crimson", ls="--", label="Vertical: biggest machine's ceiling")
ax.set_xlabel("Hour of day"); ax.set_ylabel("Users"); ax.set_title("Launch day")
ax.legend(fontsize=8); plt.show()

turned_away = int((users - vertical_served).sum())
print(f"Vertical: during the launch, one huge machine turns away {turned_away:,} user-visits (hourly totals).")
print(f"Horizontal: scales from 1 to {servers_needed.max()} servers and back. Nobody turned away.")

### What does that cost?

In [ ]:
plans = {
    "Buy enough for the peak, keep it all day": int(servers_needed.max()) * 24,
    "Autoscale: add and remove servers each hour": int(servers_needed.sum()),
}
for plan, server_hours in plans.items():
    print(f"{plan:<46} {server_hours:>4} server-hours")
saving = 1 - plans["Autoscale: add and remove servers each hour"] / plans["Buy enough for the peak, keep it all day"]
print(f"\nAutoscaling uses {saving:.0%} fewer server-hours for the same day.")

---
## Chapter 4 — When the cloud bites back  *(33–38 min)*

### The forgotten GPU
Friday 5pm: you train a model on a rented GPU cluster, close your laptop and go home.

In [ ]:
GPU_CLUSTER_PER_HOUR = 30.00    # made-up price for 8 GPUs
hours_forgotten = 64            # Friday 5pm → Monday 9am
bill = GPU_CLUSTER_PER_HOUR * hours_forgotten
print(f"Weekend bill for doing nothing: ${bill:,.0f}")

def budget_alert(spent, budget):
    used = spent / budget
    if used >= 1.0:
        print(f"🚨 OVER BUDGET: {used:.0%} of your ${budget:,} budget is gone")
    elif used >= 0.8:
        print(f"⚠️  Warning: {used:.0%} of your ${budget:,} budget used")
    else:
        print(f"OK: {used:.0%} of budget used")

budget_alert(spent=bill, budget=1_000)

### Sometimes your laptop is the right answer
How big is our whole 3-year weather dataset?

In [ ]:
rng = np.random.default_rng(0)
dates = pd.date_range("2023-01-01", periods=1096, freq="D")
wave = -np.cos(2 * np.pi * (dates.dayofyear.values - 15) / 365)
weather = pd.DataFrame({
    "date": dates.strftime("%Y-%m-%d"),
    "temp_max": (17 + 9 * wave + rng.normal(0, 3.5, 1096)).round(1),
    "rain_mm": np.where(rng.random(1096) < 0.3, rng.gamma(1.2, 4, 1096), 0).round(1),
    "wind_kmh": np.clip(rng.normal(17, 8, 1096), 2, None).round(1),
})
size_kb = len(weather.to_csv(index=False).encode()) / 1024
print(f"{len(weather):,} days of weather = {size_kb:.0f} KB")
print(f"A phone photo is ~3,000 KB. Our whole dataset is smaller than one photo.")

---
## Chapter 5 — The provider landscape  *(38–46 min)*

**The big three "hyperscalers"** (planet-scale providers):

| Provider | Known for |
|---|---|
| **AWS** (Amazon) | Oldest and largest, broadest catalogue, default for many startups |
| **GCP** (Google) | Data and ML strengths (BigQuery, Vertex AI) |
| **Azure** (Microsoft) | Enterprise and government, Microsoft 365 integration |

**Beyond them:** developer platforms (DigitalOcean, Render, Fly.io, Cloudflare) are simpler, with fewer services.
**Backend-as-a-Service:** Firebase, Supabase give you a ready backend (database, auth, storage) through an API.

All the big three offer roughly the same *kinds* of service under different names. Learn the **categories**, not the names:

In [ ]:
taxonomy = pd.DataFrame([
    ["Compute",            "Run code on a server",          "EC2",        "Compute Engine",  "Virtual Machines", "—"],
    ["Serverless",         "Run functions, no servers",     "Lambda",     "Cloud Functions", "Azure Functions",  "Edge Functions"],
    ["Object storage",     "Store files by key",            "S3",         "Cloud Storage",   "Blob Storage",     "Storage"],
    ["Managed relational DB", "Hosted SQL database",        "RDS",        "Cloud SQL",       "Azure SQL",        "✓ core product"],
    ["LLM API",            "Use large language models",     "Bedrock",    "Vertex AI",       "Azure OpenAI",     "—"],
    ["ML platform",        "Train and deploy models",       "SageMaker",  "Vertex AI",       "Azure ML",         "—"],
], columns=["Category", "What it does", "AWS", "GCP", "Azure", "Supabase"]).set_index("Category")
taxonomy

### Read a real-looking job description
A made-up job ad, full of service names. Let's translate the jargon into categories.

In [ ]:
job_ad = """
Data Engineer, weather analytics startup.
You'll build ETL pipelines that land raw files in S3 and load clean tables into Postgres on RDS.
Jobs are triggered by Lambda. Bonus: BigQuery experience, or deploying models with SageMaker.
We containerise everything with Docker and run services on EC2.
"""

extra = {"Postgres": "Managed relational DB", "BigQuery": "Data warehouse (analytics SQL)",
         "Docker": "Packaging (containers)"}
lookup = {name: cat for cat, row in taxonomy.iterrows() for name in row[["AWS", "GCP", "Azure"]]} | extra

found = [(name, lookup[name]) for name in lookup if re.search(rf"\b{re.escape(name)}\b", job_ad)]
pd.DataFrame(found, columns=["Mentioned in the ad", "What it actually is"]).drop_duplicates()

---
## Chapter 6 — Why Supabase: a table vs a file  *(46–56 min)*

Weeks 9–11 use **Supabase**: a hosted **PostgreSQL** database plus an auto-generated API, auth and storage.
The course moved from Azure for three reasons:

1. **Access** — you sign up yourself in two minutes. No waiting for an organisation invite.
2. **Structured data** — rows and columns you can query, not opaque files in folders.
3. **Pipeline fit** — our ETL has a *raw* zone and an *enriched* zone. In Supabase those are two related tables.

Slide 9's question: *a Supabase table and an S3 bucket can both store weather data. What can you do with the table that you can't do with the file?*
Let's find out. (We use SQLite here, a tiny local database, because it speaks almost the same SQL as Postgres.)

In [ ]:
# The FILE way: one CSV sitting in a bucket
weather.to_csv("weather_bucket.csv", index=False)

# The TABLE way: a database with a raw zone and an enriched zone
db = sqlite3.connect(":memory:")
db.execute("""CREATE TABLE weather_raw (
    date TEXT PRIMARY KEY, temp_max REAL, rain_mm REAL, wind_kmh REAL)""")
db.execute("""CREATE TABLE weather_enriched (
    date TEXT PRIMARY KEY REFERENCES weather_raw(date), season TEXT, good_day INTEGER)""")

weather.to_sql("weather_raw", db, if_exists="append", index=False)
month = pd.to_datetime(weather.date).dt.month
enriched = pd.DataFrame({
    "date": weather.date,
    "season": np.select([month.isin([12, 1, 2]), month.isin([3, 4, 5]), month.isin([6, 7, 8])],
                        ["winter", "spring", "summer"], "autumn"),
    "good_day": ((weather.temp_max.between(4, 28)) & (weather.rain_mm < 1) & (weather.wind_kmh < 24)).astype(int),
})
enriched.to_sql("weather_enriched", db, if_exists="append", index=False)
print("raw rows:", db.execute("SELECT COUNT(*) FROM weather_raw").fetchone()[0],
      "| enriched rows:", db.execute("SELECT COUNT(*) FROM weather_enriched").fetchone()[0])

### Difference 1: ask a question, get back only the answer
*"How many good walking days did each season have in 2024?"*

In [ ]:
# FILE: download the whole thing, then work it out yourself
whole_file = pd.read_csv("weather_bucket.csv")
print(f"File way: downloaded all {len(whole_file):,} rows to answer one question.\n")

# TABLE: send the question, the database does the work
answer = pd.read_sql("""
    SELECT e.season, SUM(e.good_day) AS good_days, ROUND(AVG(r.temp_max), 1) AS avg_temp
    FROM weather_raw r JOIN weather_enriched e ON r.date = e.date
    WHERE r.date LIKE '2024-%'
    GROUP BY e.season ORDER BY good_days DESC""", db)
print(f"Table way: got back {len(answer)} rows.")
answer

### Difference 2: the table protects the data
Our pipeline accidentally runs twice and tries to insert the same day again.

In [ ]:
duplicate = weather.iloc[[0]]

duplicate.to_csv("weather_bucket.csv", mode="a", header=False, index=False)
print("File: accepted the duplicate. Rows now:", len(pd.read_csv("weather_bucket.csv")))

try:
    db.execute("INSERT INTO weather_raw VALUES (?, ?, ?, ?)", tuple(duplicate.iloc[0]))
except sqlite3.IntegrityError as e:
    print("Table: refused it →", e)

### Difference 3: Supabase gives every table a web API for free
Supabase turns a URL into a SQL query, so any app can read the data over the internet.
Here's a toy version of that translation, to show the idea:

In [ ]:
OPS = {"eq": "=", "gt": ">", "lt": "<", "gte": ">=", "lte": "<="}

def url_to_sql(url):
    table, query = url.split("?")
    cols, where, limit = "*", [], ""
    for part in query.split("&"):
        key, value = part.split("=", 1)
        if key == "select":
            cols = value
        elif key == "limit":
            limit = f" LIMIT {int(value)}"
        else:
            op, val = value.split(".", 1)
            where.append(f"{key} {OPS[op]} {val if val.replace('.', '').isdigit() else repr(val)}")
    return f"SELECT {cols} FROM {table}" + (" WHERE " + " AND ".join(where) if where else "") + limit

url = "weather_raw?select=date,temp_max&temp_max=gt.30&limit=5"
sql = url_to_sql(url)
print("URL:", "/rest/v1/" + url)
print("SQL:", sql)
pd.read_sql(sql, db)

---
## Wrap-up  *(56–60 min)*

1. **Cloud = rent, don't buy.** Pay for what you use; great for spiky needs.
2. **IaaS → PaaS → SaaS:** the question is always *"what do I manage?"*
3. **Scaling:** vertical (bigger machine, has a ceiling) vs horizontal (more machines, scales on demand).
4. **Costs can spiral.** Set budget alerts, shut things down, and prototype locally when data is small.
5. **Providers:** AWS (broadest), GCP (data and ML), Azure (enterprise). Learn the **categories**, not the names.
6. **Supabase:** hosted Postgres you can sign up for yourself. A table can be queried, joined, protected and served as an API, unlike a plain file.

> *The cloud isn't one thing. It's a spectrum of services with different trade-offs.*

### Where we're heading
Weeks 9–11: our weather pipeline moves to Supabase. Raw readings go into one table, enriched data into another,
and our Week 4 classifier and Week 5–7 AI tools read from it.

### Before next week
- Create a free Supabase account at **supabase.com**.
- Think: which parts of our pipeline must run in the cloud, and which are fine on a laptop?